# ETL - Preparação e Tratamento de Dados

Este notebook tem como objetivo realizar o processo de **ETL (Extract, Transform, Load)** dos datasets utilizados no projeto SmartHealth 360.

O projeto utiliza duas fontes de dados distintas:

1. **Dataset hospitalar real (Mater Dei)**  
   Utilizado para análises exploratórias e entendimento do contexto hospitalar.

2. **Dataset público (Kaggle - Medical Appointment No Shows)**  
   Utilizado para o treinamento do modelo de Machine Learning responsável por prever a probabilidade de ausência em consultas médicas.

Devido às diferenças estruturais entre os datasets, cada um passa por um processo de ETL independente.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

### ETL - Dataset Hospitalar Mater Dei

Este dataset contém informações reais sobre consultas e pacientes do Hospital Mater Dei.

Objetivos do tratamento:

- corrigir tipos de dados
- tratar valores nulos
- selecionar colunas relevantes
- preparar os dados para análise exploratória

In [2]:
# Carregar Dataset
materdei = pd.read_csv(
    "../data/raw/materdei.csv",
    encoding="latin1",
    sep=";"
)

In [3]:
# Visualização inicial
materdei.head()

,CD_MULTI_EMPRESA,DS_MULTI_EMPRESA,COMPETENCIA,ANO,TIPO_AGENDA,USUARIO,CLUSTERR,CD_IT_AGENDA_CENTRAL,CD_PACIENTE,DT_NASCIMENTO,IDADE,CD_UNIDADE_ATENDIMENTO,DS_UNIDADE_ATENDIMENTO
0,6,6-HOSPITAL MATER DEI S/A CONTORNO,5/1/2025,2025,AGENDA IMAGEM,CENTRAL DE MARCAÇÕES,OUTRAS PRACAS,111668221,805894,11/9/1948 2:00 AM,76.0,2.0,UNIDADE CONTORNO
1,6,6-HOSPITAL MATER DEI S/A CONTORNO,5/1/2025,2025,AGENDA IMAGEM,CENTRAL DE MARCAÇÕES,OUTRAS PRACAS,113118802,505908,5/16/1978 2:00 AM,47.0,2.0,UNIDADE CONTORNO
2,6,6-HOSPITAL MATER DEI S/A CONTORNO,5/1/2025,2025,AGENDA IMAGEM,CENTRAL DE MARCAÇÕES,OUTRAS PRACAS,103715292,49339,12/3/1979 2:00 AM,45.0,2.0,UNIDADE CONTORNO
3,6,6-HOSPITAL MATER DEI S/A CONTORNO,5/1/2025,2025,AGENDA AMBULATORIAL,CENTRAL DE MARCAÇÕES,OUTRAS PRACAS,90686184,2132945,2/20/1960 2:00 AM,65.0,2.0,UNIDADE CONTORNO
4,1,1-HOSPITAL MATER DEI S/A STO AGOSTINHO,5/1/2025,2025,AGENDA AMBULATORIAL,CENTRAL DE MARCAÇÕES,OUTRAS PRACAS,114852768,2165364,2/1/1991 2:00 AM,34.0,43.0,MAIS SAÚDE


In [4]:
# Estrutura dos dados
materdei.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279304 entries, 0 to 279303
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   CD_MULTI_EMPRESA        279304 non-null  int64  
 1   DS_MULTI_EMPRESA        279304 non-null  object 
 2   COMPETENCIA             279304 non-null  object 
 3   ANO                     279304 non-null  int64  
 4   TIPO_AGENDA             279304 non-null  object 
 5   USUARIO                 279304 non-null  object 
 6   CLUSTERR                279304 non-null  object 
 7   CD_IT_AGENDA_CENTRAL    279304 non-null  int64  
 8   CD_PACIENTE             279304 non-null  int64  
 9   DT_NASCIMENTO           279222 non-null  object 
 10  IDADE                   279222 non-null  float64
 11  CD_UNIDADE_ATENDIMENTO  278524 non-null  float64
 12  DS_UNIDADE_ATENDIMENTO  278524 non-null  object 
dtypes: float64(2), int64(4), object(7)
memory usage: 27.7+ MB


In [5]:
# Tratamento de datas
materdei["DT_NASCIMENTO"] = pd.to_datetime(
    materdei["DT_NASCIMENTO"],
    errors="coerce"
)

# Remover valores inválidos de idade
materdei = materdei.dropna(subset=["IDADE"])

# Converter idade para inteiro
materdei["IDADE"] = materdei["IDADE"].astype(int)

# Colunas relevantes
materdei_clean = materdei[
    [
        "CD_PACIENTE",
        "IDADE",
        "TIPO_AGENDA",
        "DS_UNIDADE_ATENDIMENTO",
        "COMPETENCIA"
    ]
]

# Salvar dataset tratado
materdei_clean.to_csv(
    "../data/processed/materdei_clean.csv",
    index=False
)

# ETL - Dataset Kaggle (Medical Appointment No Shows)

Este dataset contém registros de consultas médicas e indica se o paciente compareceu ou não.

Objetivo do tratamento:

- preparar os dados para treinamento do modelo de Machine Learning
- converter variáveis categóricas
- criar novas features relevantes

In [6]:
# Carregar dataset
kaggle = pd.read_csv("../data/raw/dataset_kaggle.csv")

In [7]:
df = pd.read_csv("../data/raw/dataset_kaggle.csv")

print("Dimensão inicial do dataset:")
print(df.shape)

df.head()

Dimensão inicial do dataset:
(110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [8]:
# Diagnóstico inicial dos dados
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  object 
 3   ScheduledDay    110527 non-null  object 
 4   AppointmentDay  110527 non-null  object 
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  object 
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  object 
dtypes: float64(1), int64(8), object(5)
memory usage: 11.8+ MB


,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,1.105270e+05,1.105270e+05,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000
mean,1.474963e+14,5.675305e+06,37.088874,0.098266,0.197246,0.071865,0.030400,0.022248,0.321026
std,2.560949e+14,7.129575e+04,23.110205,0.297675,0.397921,0.258265,0.171686,0.161543,0.466873
min,3.921784e+04,5.030230e+06,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.172614e+12,5.640286e+06,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.173184e+13,5.680573e+06,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.439172e+13,5.725524e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,9.999816e+14,5.790484e+06,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000


In [9]:
# Padronização e tradução das colunas
df = df.rename(columns={
    "PatientId": "id_paciente",
    "AppointmentID": "id_consulta",
    "Gender": "genero",
    "ScheduledDay": "data_agendamento",
    "AppointmentDay": "data_consulta",
    "Age": "idade",
    "Neighbourhood": "bairro",
    "Scholarship": "bolsa_familia",
    "Hipertension": "hipertensao",
    "Diabetes": "diabetes",
    "Alcoholism": "alcoolismo",
    "Handcap": "deficiencia",
    "SMS_received": "sms_recebido",
    "No-show": "no_show"
})

In [10]:
# Conversão de variáveis de data
df["data_agendamento"] = pd.to_datetime(df["data_agendamento"]).dt.date
df["data_consulta"] = pd.to_datetime(df["data_consulta"]).dt.date

In [11]:
# Feature Engineering
## Criar dias de espera
df["dias_espera"] = (
    pd.to_datetime(df["data_consulta"]) -
    pd.to_datetime(df["data_agendamento"])
).dt.days

## Criar variáveis temporais
df["dia_semana_consulta"] = pd.to_datetime(df["data_consulta"]).dt.dayofweek

df["mes_consulta"] = pd.to_datetime(df["data_consulta"]).dt.month

# Consulta no mesmo dia
df["consulta_mesmo_dia"] = (df["dias_espera"] == 0).astype(int)

In [12]:
# Verificar problemas
print("Idades negativas:")
print((df["idade"] < 0).sum())

print("\nDias de espera negativos:")
print((df["dias_espera"] < 0).sum())

Idades negativas:
1

Dias de espera negativos:
5


In [13]:
# Limpeza de dados inconsistentes
df = df[df["idade"] >= 0]

df = df[df["dias_espera"] >= 0]

In [14]:
## TARGET
# Converter variável alvo
df["no_show"] = df["no_show"].map({
    "No": 0,
    "Yes": 1
})

In [15]:
# Remoção de colunas irrelevantes
df = df.drop(
    ["id_paciente", "id_consulta"],
    axis=1
)

In [16]:
# Codificação de variáveis categóricas
df = pd.get_dummies(
    df,
    columns=["genero", "bairro"],
    drop_first=True
)

# Converter booleanos para inteiros
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

# Resetar index
df = df.reset_index(drop=True)

In [17]:
# Diagnóstico pós-tratamento
## Verificação final da estrutura
print("Dimensão final do dataset:")
print(df.shape)

df.info()

## Verificação final de valores nulos
df.isnull().sum()

Dimensão final do dataset:
(110521, 95)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110521 entries, 0 to 110520
Data columns (total 95 columns):
 #   Column                              Non-Null Count   Dtype 
---  ------                              --------------   ----- 
 0   data_agendamento                    110521 non-null  object
 1   data_consulta                       110521 non-null  object
 2   idade                               110521 non-null  int64 
 3   bolsa_familia                       110521 non-null  int64 
 4   hipertensao                         110521 non-null  int64 
 5   diabetes                            110521 non-null  int64 
 6   alcoolismo                          110521 non-null  int64 
 7   deficiencia                         110521 non-null  int64 
 8   sms_recebido                        110521 non-null  int64 
 9   no_show                             110521 non-null  int64 
 10  dias_espera                         110521 non-null  int64 
 11 

data_agendamento        0
data_consulta           0
idade                   0
bolsa_familia           0
hipertensao             0
                       ..
bairro_SÃO JOSÉ         0
bairro_SÃO PEDRO        0
bairro_TABUAZEIRO       0
bairro_UNIVERSITÁRIO    0
bairro_VILA RUBIM       0
Length: 95, dtype: int64

In [18]:
print("Dimensão final:", df.shape)

print("\nTipos de dados:")
print(df.dtypes.value_counts())

print("\nValores nulos:")
print(df.isnull().sum().sum())

Dimensão final: (110521, 95)

Tipos de dados:
int64     91
object     2
int32      2
Name: count, dtype: int64

Valores nulos:
0


In [19]:
df.to_csv(
    "../data/processed/kaggle_tratado.csv",
    index=False
)